# Project 43 — Explainable Disaster Severity Assessment
## Notebook 04: Dashboard Development & Testing
**C-DAC Mohali | ML to Generative AI & LLMs**
Team: Anuksha | Rishika | Ipshita
Dataset: AIDERv2 | Backbone: EfficientNet (auto-detected from `models/best_model.h5`)

Development/testing notebook for `app/app.py` (the Streamlit UI). Exercises
`utils/predict.py` and `utils/gradcam.py` exactly as the app calls them, so
issues surface here — where they're easy to debug — before they surface in
the browser.

## Setup
Same `find_project_root()` pattern as notebooks 01–03.

In [ ]:
import os, sys, glob, warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

def find_project_root(marker="data", max_up=5):
    """Walk up from cwd until a directory containing `marker/` is found."""
    cur = Path.cwd().resolve()
    for _ in range(max_up + 1):
        if (cur / marker).is_dir():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise FileNotFoundError(
        f"Could not locate project root containing '{marker}/' "
        f"starting from {Path.cwd()}"
    )

PROJECT_ROOT = find_project_root("data")
print(f"Project root: {PROJECT_ROOT}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.predict import load_model, preprocess_image, predict, CLASS_NAMES, LOW_CONFIDENCE_THRESHOLD
from utils.gradcam import (
    get_gradcam_heatmap,
    get_gradcam_plus_plus_heatmap,
    overlay_heatmap,
    predict_and_explain,
)

TEST_DIR = str(PROJECT_ROOT / "data" / "Test")
MODELS_DIR = PROJECT_ROOT / "models"

MODEL = None
MODEL_PATH = MODELS_DIR / "best_model.h5"
if MODEL_PATH.is_file():
    MODEL = load_model(str(MODEL_PATH))
    print(f"Loaded {MODEL_PATH}")
else:
    fallbacks = sorted(MODELS_DIR.glob("best_model_*.h5"), key=lambda p: p.stat().st_mtime, reverse=True)
    if fallbacks:
        MODEL_PATH = fallbacks[0]
        MODEL = load_model(str(MODEL_PATH))
        print(
            f"models/best_model.h5 not found — using fallback checkpoint "
            f"'{MODEL_PATH.name}'.\nRun notebook 02's promotion cell for "
            "the official artifact."
        )
    else:
        print(
            "No trained model found in models/. Run "
            "notebooks/02_model_training.ipynb first — this notebook needs "
            "a trained model to exercise the predict/gradcam pipeline."
        )

if MODEL is not None:
    IMG_SIZE = MODEL.input_shape[1:3]

def any_test_image(class_name=None):
    """Return a path to a test image, optionally from a specific class."""
    classes = [class_name] if class_name else list(CLASS_NAMES)
    for cls in classes:
        files = (glob.glob(os.path.join(TEST_DIR, cls, '*.jpg')) +
                 glob.glob(os.path.join(TEST_DIR, cls, '*.jpeg')) +
                 glob.glob(os.path.join(TEST_DIR, cls, '*.png')))
        if files:
            return files[0]
    return None

## 1. Test `predict.py` pipeline end-to-end
Loads the model, picks a test image, runs `predict()`, and confirms the
confidence dict has the shape `app.py` expects: `{class_name: float, ...}`
plus an optional `'warning'` key.

In [ ]:
if MODEL is not None:
    img_path = any_test_image()
    print("Test image:", img_path)

    img_array = preprocess_image(img_path, target_size=IMG_SIZE)
    class_label, confidence_dict = predict(MODEL, img_array)

    print("Predicted class:", class_label)
    print("Confidence dict:", confidence_dict)

    # app.py expects exactly CLASS_NAMES as keys (+ optional 'warning')
    expected_keys = set(CLASS_NAMES)
    actual_keys = set(k for k in confidence_dict if k != 'warning')
    assert actual_keys == expected_keys, f"Mismatch: {actual_keys} vs {expected_keys}"
    print("\nconfidence_dict format matches what app.py expects.")
else:
    print("Skipping — no model available. Run notebook 02 first.")

## 2. Test `gradcam.py` pipeline end-to-end
Same image, both heatmaps, both overlays — the exact 3-panel layout
`app.py` renders (Original | Grad-CAM | Grad-CAM++).

In [ ]:
if MODEL is not None:
    class_idx = CLASS_NAMES.index(class_label)
    original_uint8 = np.uint8(np.clip(img_array[0], 0, 1) * 255)

    heatmap = get_gradcam_heatmap(MODEL, img_array, class_idx)
    heatmap_pp = get_gradcam_plus_plus_heatmap(MODEL, img_array, class_idx)
    _, overlay_rgb = overlay_heatmap(heatmap, original_uint8)
    _, overlay_pp_rgb = overlay_heatmap(heatmap_pp, original_uint8)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(original_uint8); axes[0].set_title("Original"); axes[0].axis('off')
    axes[1].imshow(overlay_rgb); axes[1].set_title("Grad-CAM"); axes[1].axis('off')
    axes[2].imshow(overlay_pp_rgb); axes[2].set_title("Grad-CAM++"); axes[2].axis('off')
    plt.suptitle(f"Predicted: {class_label}")
    plt.tight_layout()
    plt.show()

    # Also exercise the single convenience wrapper app.py calls directly.
    result = predict_and_explain(MODEL, img_path)
    print("predict_and_explain() keys:", list(result.keys()))
else:
    print("Skipping — no model available. Run notebook 02 first.")

## 3. Test confidence thresholding
Examples of high (>80%), medium (60–80%), and low (<60%) confidence
predictions, and verification that `predict()`'s `'warning'` key triggers
exactly at `LOW_CONFIDENCE_THRESHOLD`.

In [ ]:
if MODEL is not None:
    buckets = {"high (>80%)": [], "medium (60-80%)": [], "low (<60%)": []}

    all_test_files = []
    for cls in CLASS_NAMES:
        all_test_files.extend(
            glob.glob(os.path.join(TEST_DIR, cls, '*.jpg')) +
            glob.glob(os.path.join(TEST_DIR, cls, '*.jpeg')) +
            glob.glob(os.path.join(TEST_DIR, cls, '*.png'))
        )

    # Sample a modest number of images to bucket by confidence.
    import random
    random.seed(42)
    sample_files = random.sample(all_test_files, min(30, len(all_test_files)))

    for f in sample_files:
        arr = preprocess_image(f, target_size=IMG_SIZE)
        label, conf = predict(MODEL, arr)
        top_prob = conf[label]
        if top_prob > 0.80:
            bucket = "high (>80%)"
        elif top_prob >= 0.60:
            bucket = "medium (60-80%)"
        else:
            bucket = "low (<60%)"
        buckets[bucket].append((f, label, top_prob, 'warning' in conf))

    for name, items in buckets.items():
        print(f"\n{name}: {len(items)} example(s)")
        for f, label, top_prob, has_warning in items[:3]:
            print(f"  {os.path.basename(f):20s} -> {label:12s} {top_prob:.1%}  warning={has_warning}")

    # Verify the warning flag triggers exactly below LOW_CONFIDENCE_THRESHOLD.
    for f, label, top_prob, has_warning in [item for items in buckets.values() for item in items]:
        expected_warning = top_prob < LOW_CONFIDENCE_THRESHOLD
        assert has_warning == expected_warning, f"Warning flag mismatch for {f}"
    print(f"\nWarning flag correctly triggers below {LOW_CONFIDENCE_THRESHOLD:.0%} for all {len(sample_files)} sampled images.")
else:
    print("Skipping — no model available. Run notebook 02 first.")

## 4. Streamlit app walkthrough

### Launching the app
```bash
python -m streamlit run app/app.py
```
This opens the dashboard at `http://localhost:8501`. `app.py` loads
`models/best_model.h5` on first use (`st.cache_resource`), so make sure
notebook 02's promotion cell has run first — otherwise the app shows a clear
error and stops rather than crashing.

### Features to test manually
- [ ] **Upload flow** — upload a PNG/JPG and confirm prediction + confidence appear.
- [ ] **Sample image buttons** — sidebar buttons for each class load a
      `data/Test/{class}/` image without needing a manual upload.
- [ ] **3-column Grad-CAM display** — Original | Grad-CAM overlay | Grad-CAM++
      overlay all render after a prediction.
- [ ] **Confidence bar chart** — all 4 class probabilities shown as horizontal
      bars, colored green (>80%), yellow (60–80%), red (<60%).
- [ ] **Confidence threshold slider** — sidebar slider (40%–80%, default 60%);
      moving it changes when the low-confidence warning appears.
- [ ] **Low-confidence warning** — predictions below the slider threshold show
      `⚠️ Low confidence — human review recommended`.
- [ ] **Model info panel** — sidebar shows which backbone variant is loaded
      (B0/B1/B3), parameter count, and input size.
- [ ] **Missing-model handling** — temporarily rename `models/best_model.h5`
      and confirm the app shows a clear error via `st.error()` + `st.stop()`
      instead of a stack trace.